# Notebook 3: Inference — Prompting & Chatting with the Fine-Tuned Model

Two ways to use the SFT model at inference time:

1. **Single-shot prompting** — construct the full prompt (system + context + history),
   generate one agent response. Good for batch evaluation, A/B testing, or integrating
   into a pipeline.
2. **Interactive chat** — loop where the model generates agent turns and you type
   customer turns. Good for qualitative testing.

### Why we use our structured prompt, not `apply_chat_template`

The model was trained on our custom special-token format:
```
<|system|>...<|/system|>
<|context|>...<|/context|>
<|conversation|>
<|agent|>...
<|customer|>...
<|/conversation|>
<|agent|>[generates here]
```

A generic chat template (`apply_chat_template`) would produce different tokens
(`<|im_start|>user`, `[INST]`, etc.) that the model never saw during SFT.
Using our format ensures the model's attention heads activate on the positions
they were trained on.

In [ ]:
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ============================================================
# Configuration
# ============================================================
MODEL_DIR = "/path/to/gpt-oss-120b"          # <-- FILL IN: base model checkpoint
ADAPTER_PATH = "checkpoints/final_adapter"    # LoRA adapter from notebook 2

# Fixed system prompt — must match training
SYSTEM_PROMPT = (
    "You are an outbound sales agent for American Express.\n"
    "Your goal is to identify customer needs and guide the conversation\n"
    "toward [product/outcome]. Use information provided in the context\n"
    "block when available to reference relevant offers, campaigns,\n"
    "or product details. If no context is provided, rely on the\n"
    "conversation history to guide your response."
)

## Load Base Model + LoRA Adapter

In [ ]:
# Load tokenizer from the adapter dir (has the 8 registered special tokens)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, local_files_only=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
base_model.resize_token_embeddings(len(tokenizer))

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

# Token IDs used for stop conditions
STOP_TOKEN_IDS = [
    tokenizer.convert_tokens_to_ids("<|customer|>"),
    tokenizer.convert_tokens_to_ids("<|/conversation|>"),
    tokenizer.eos_token_id,
]

print("Model + adapter loaded. Ready for inference.")

## Prompt Builder\n\nThis is the core utility — it assembles our structured prompt from components.\nBoth single-shot prompting and interactive chat use this same function.\n\n```\n<|system|>{system_prompt}<|/system|>\n<|context|>{context}<|/context|>\n<|conversation|>\n<|agent|>{turn 1}\n<|customer|>{turn 2}\n...\n<|/conversation|>\n<|agent|>                    ← model generates from here\n```

In [ ]:
def build_prompt(
    conversation_turns: list[tuple[str, str]],
    context: str = "",
    system_prompt: str = SYSTEM_PROMPT,
) -> str:
    """Build the structured prompt string for agent-turn generation.

    Args:
        conversation_turns: List of (speaker, text) tuples.
            speaker is "agent" or "customer".
            Can be empty — the model will generate the opening line.
        context: Optional RAG context (product info, campaign details).
            Empty string = no context (model relies on conversation history).
        system_prompt: The system instruction block.

    Returns:
        Prompt string ending with <|agent|>, ready for generation.
    """
    # Conversation history
    conv_lines = []
    for speaker, text in conversation_turns:
        marker = "<|agent|>" if speaker == "agent" else "<|customer|>"
        conv_lines.append(f"{marker}{text}")
    conv_str = "\n".join(conv_lines)

    prompt = (
        f"<|system|>{system_prompt}<|/system|>\n"
        f"<|context|>{context}<|/context|>\n"
        f"<|conversation|>\n"
        f"{conv_str}\n"
        f"<|/conversation|>\n"
        f"<|agent|>"
    )
    return prompt


def generate(
    prompt: str,
    max_new_tokens: int = 256,
    temperature: float = 0.7,
    top_p: float = 0.9,
) -> str:
    """Generate text from a prompt, stopping at turn boundaries.

    Returns only the agent's response text (no special tokens).
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            eos_token_id=STOP_TOKEN_IDS,
        )

    generated_ids = output_ids[0][input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=False)

    # Truncate at the first special token (model might bleed past its turn)
    for stop_tok in ["<|customer|>", "<|/conversation|>", "<|agent|>",
                     tokenizer.eos_token]:
        if stop_tok in response:
            response = response[:response.index(stop_tok)]

    return response.strip()

---

# Part 1: Single-Shot Prompting

Construct the full prompt manually, call `generate()`, get one agent response.
This is what you'd use in a production pipeline or for batch evaluation.

### Example 1a: Cold open — no conversation history\n\nThe agent generates the very first line of an outbound call.\nThis tests whether the model learned good opening patterns.

In [ ]:
# No history, no context — pure cold open
prompt = build_prompt(conversation_turns=[])

print("=== PROMPT ===")
print(prompt)
print()

response = generate(prompt)
print("=== AGENT OPENS WITH ===")
print(response)

### Example 1b: Cold open with RAG context\n\nSame scenario, but now the `<|context|>` block has product info.\nThe model should weave this into its opening.

In [ ]:
prompt = build_prompt(
    conversation_turns=[],
    context="Product: Business Platinum Card\nOffer: 150K MR Points Sign-Up Bonus",
)

print("=== PROMPT ===")
print(prompt)
print()

response = generate(prompt)
print("=== AGENT OPENS WITH (context-aware) ===")
print(response)

### Example 1c: Mid-conversation — handle an objection\n\nProvide a few turns of history ending with a customer objection.\nThe model should generate the agent's response to that specific objection.

In [ ]:
history = [
    ("agent", "Hi, this is Sarah from American Express. I'm reaching out because "
              "we noticed your company's travel spend has been growing and I wanted "
              "to share some options that could help you get more value from that."),
    ("customer", "Yeah, we already have a corporate card program with Chase. "
                 "Why would we switch?"),
]

prompt = build_prompt(conversation_turns=history)

response = generate(prompt)
print("=== CONVERSATION SO FAR ===")
for speaker, text in history:
    label = "Agent" if speaker == "agent" else "Customer"
    print(f"{label}: {text}")
print()
print("=== AGENT RESPONDS ===")
print(f"Agent: {response}")

### Example 1d: Batch evaluation — generate N responses for the same prompt\n\nUseful for measuring response diversity and consistency.

In [ ]:
# Reuse the objection-handling prompt from 1c
prompt = build_prompt(conversation_turns=history)

print("Generating 5 different responses to the same objection:\n")
for i in range(5):
    resp = generate(prompt, temperature=0.8)
    print(f"  [{i+1}] {resp}")
    print()

---

# Part 2: Interactive Chat

A loop where the model plays the agent and you play the customer.
The model generates the first line (outbound call), then you respond.

In [ ]:
def chat(context: str = "", max_turns: int = 20):
    """Interactive outbound sales call simulation.

    Args:
        context: Optional RAG context to populate the <|context|> block.
            Example: "Product: Business Gold Card\\nCampaign: Q2 Spend Bonus"
        max_turns: Safety limit on conversation length.
    """
    conversation_turns = []

    print("=" * 60)
    print("OUTBOUND SALES CALL SIMULATION")
    print("You are the customer. The agent will speak first.")
    if context:
        print(f"Context: {context}")
    print("Type 'quit' to end  |  'history' to see full transcript")
    print("=" * 60)

    for turn_num in range(max_turns):
        # Build prompt and generate agent turn
        prompt = build_prompt(conversation_turns, context=context)
        agent_text = generate(prompt)

        print(f"\nAgent: {agent_text}")
        conversation_turns.append(("agent", agent_text))

        # Get customer input
        user_input = input("\nCustomer: ").strip()

        if user_input.lower() == "quit":
            print("\n[Call ended]")
            break
        elif user_input.lower() == "history":
            print("\n--- FULL TRANSCRIPT ---")
            for i, (spk, txt) in enumerate(conversation_turns):
                label = "Agent" if spk == "agent" else "Customer"
                print(f"  {label}: {txt}")
            print("--- END ---")
            # Re-prompt for actual input
            user_input = input("\nCustomer: ").strip()
            if user_input.lower() == "quit":
                print("\n[Call ended]")
                break

        conversation_turns.append(("customer", user_input))

    return conversation_turns

### Chat without RAG context

In [ ]:
# No context — agent relies on conversation flow alone
transcript = chat()

### Chat with RAG context\n\nThis previews how Layer 2 (Product RAG) will work at inference time.\nThe context block is populated with product details that the agent should reference.

In [ ]:
# With context — agent should incorporate product details into responses
transcript_with_ctx = chat(
    context="Product: Business Platinum Card\n"
            "Offer: 150K MR Points Sign-Up Bonus\n"
            "Annual Fee: $695 (waived first year)\n"
            "Key Benefit: 5X points on flights and prepaid hotels booked through amextravel.com"
)

---

## Summary: Prompting vs Chat Template

| Approach | When to use | Our case |
|---|---|---|
| **Our structured prompt** (`build_prompt`) | Model was SFT'd on custom special tokens | **Use this** |
| `apply_chat_template` | Model was trained with a standard chat format (ChatML, Llama-chat) | Don't use |
| Raw text prompting (no special tokens) | Base model with no fine-tuning | Only for notebook 0 (fake data gen) |

The key insight: **the prompt format at inference must match the format at training time**.
Our model saw `<|system|>`, `<|agent|>`, `<|customer|>`, etc. during SFT, so that's
exactly what we feed it at inference. Using a different chat template would produce
token sequences the model wasn't trained on.